**Human Movement Intelligence | Gait Analysis | Presented by: Reza Saadatyar (2026)**

**02 | Alignment and Event Reference**

![Alignment and Event Reference](../../images/02_alignment_and_event_reference.svg)

**Objective:** Construct independently force-derived contact events from same-trial original-time sensor streams and establish participant roles.

**Dataset:** Camargo et al. (2021), treadmill/level-ground walking where original-time IMU and independent force reference are usable.

**Notebook scope:** Implemented analysis code; original-data results will be added only after execution and source eligibility confirmation.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


/content/human-movement-intelligence


**01. Imports and Research Environment**

Use the established shared utilities and the Camargo-specific alignment/reference functions. No source download or model training is triggered by importing the study.


In [2]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("gait_analysis")
NOTEBOOK_NUMBER = 2

# ----------------------- 2. MATLAB Table Reader Setup ------------------------
import importlib.util
import sys

if importlib.util.find_spec("matio") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "mat-io==1.0.2"],
        check=True,
    )

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import os
import re
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# ------------------------ 5. Study Research Utilities ------------------------
from camargo_source import read_sensor_table, validate_timebase
from gait_alignment import mapped_series, validated_overlap
from gait_reference import force_contact_events
from gait_experiments import save_prepared_trial
from gait_evaluation import participant_roles, development_folds

# ---------------------- 6. Source and Output Locations -----------------------
study_dir = root / "gait_analysis"
source_root = Path(os.environ.get("CAMARGO_DATA_ROOT",
                                  root / "data/raw/camargo"))
output_dir = study_dir / "output"
cache_dir = root / "data/processed/camargo/paired_trials"
output_dir.mkdir(parents=True, exist_ok=True)


Local changes kept; continuing with the current checkout. Conflicting Git update deferred.


**02. Audit Original Channels and Source Timing**

The previous executed Notebook 01 output contains **AB06–AB08 and nine candidate treadmill IMU/force pairs**. The revised Notebook 01 can audit the `Header` vectors of **every** source pair; rerun its Sections 05–07 to produce `01_trial_timebase_audit.csv` before continuing.

This section now opens the original channel and timing evidence and writes `02_source_contract_review.csv`. It reports which physical settings are supported by recorded field names and which require an independent source definition or additional quality control. It does **not** raise the former generic error at the start.

The original [Camargo README](https://data.mendeley.com/datasets/fcgm3chfff/1) identifies `Header` as the sample timestamp used to synchronise sensor streams and nominal sampling frequencies of 200 Hz for IMU and 1,000 Hz for force-platform recordings. Source timestamps and sample rates support *checking* a common clock, but they do not alone verify force sign, anatomical side or event-level timing precision.

The candidate fields shown here are **not approved values**. Section 03 retains its essential scientific boundary and must not be executed for force-contact labels until the source-specific contract has been reviewed.


In [3]:
# ==================== 02. Review Original Sensor Contract ====================

# ------------- 1. Load Recorded Source Pair and Channel Evidence -------------
pair_file = output_dir / "01_candidate_pairs.csv"
column_file = output_dir / "01_observed_columns.csv"
timebase_file = output_dir / "01_trial_timebase_audit.csv"
if not pair_file.is_file() or not column_file.is_file():
    raise FileNotFoundError(
        "Notebook 01's candidate pairs and observed columns are missing. "
        "Run Notebook 01 Sections 03–07 before inspecting references."
    )
pairs = pd.read_csv(pair_file)
pairs = pairs.loc[
    pairs.candidate_pair & pairs.condition.eq("treadmill")
].copy()
observed = pd.read_csv(column_file)
if pairs.empty:
    raise ValueError("No source-matched treadmill IMU/force trial pairs.")
imu_names = set(observed.loc[observed.modality == "imu", "column"])
fp_names = set(observed.loc[observed.modality == "fp", "column"])
if timebase_file.is_file():
    timebase_audit = pd.read_csv(timebase_file)
    if len(timebase_audit) != len(pairs):
        print("Trial-level timestamp audit is incomplete; rerun "
              "Notebook 01 Sections 05–07 to refresh the evidence.")
else:
    timebase_audit = pd.DataFrame()
    print("Trial-level timestamp audit missing. Rerun updated "
          "Notebook 01 Section 05 before reference extraction.")
print("Source-paired people:", pairs.subject_id.nunique())
print("Candidate source pairs:", len(pairs))
print("Gyroscope fields:", sorted(
    name for name in imu_names if "Gyro" in name
))
print("Treadmill force fields:", sorted(
    name for name in fp_names
    if "Treadmill_" in name and "_v" in name
))
if not timebase_audit.empty:
    display(timebase_audit)


# ------------------- 2. Save Scientific Contract Evidence --------------------
contract_review = pd.DataFrame([
    {"setting": "GYRO_COLUMN", "candidate": "foot_Gyro_Y",
     "observed": "foot_Gyro_Y" in imu_names,
     "unresolved": "Verify the selected gyroscope axis and instrumented side."},
    {"setting": "VERTICAL_FORCE_COLUMN",
     "candidate": "Treadmill_R_vy",
     "observed": "Treadmill_R_vy" in fp_names,
     "unresolved": "Verify physical vertical axis, source units and belt side."},
    {"setting": "FOOT_SIDE", "candidate": "right (candidate)",
     "observed": pd.NA,
     "unresolved": "Confirm physical IMU foot side and force-plate R/L mapping."},
    {"setting": "FORCE_UPWARD_SIGN", "candidate": "positive (candidate)",
     "observed": pd.NA,
     "unresolved": "Verify the real loading polarity and swing-phase baseline."},
    {"setting": "TIME_UNIT_IN_SECONDS", "candidate": "1.0 (candidate)",
     "observed": pd.NA,
     "unresolved": "Confirm Header time units with original 200/1000-Hz rates."},
    {"setting": "IMU_FORCE_TIMEBASE_CONFIRMED",
     "candidate": "not confirmed", "observed": pd.NA,
     "unresolved": "Confirm a common event-level clock for retained trials."},
])
display(contract_review)
contract_review.to_csv(
    output_dir / "02_source_contract_review.csv", index=False
)


# ------------ 3. Keep Physical Contract Unverified Until Reviewed ------------
# These are intentionally unset; source column names are not proof of units.
GYRO_COLUMN = None
VERTICAL_FORCE_COLUMN = None
FOOT_SIDE = None
FORCE_UPWARD_SIGN = None
TIME_UNIT_IN_SECONDS = None
IMU_FORCE_TIMEBASE_CONFIRMED = False
REFERENCE_THRESHOLD_N = 20.0
ANALYSED_CONDITIONS = {"treadmill"}
print("Descriptive source inspection complete. Contact-event extraction "
      "is still blocked pending physical channel and timebase verification.")
print("Evidence saved to:", output_dir / "02_source_contract_review.csv")


Source-paired people: 6
Candidate source pairs: 18
Gyroscope fields: ['foot_Gyro_X', 'foot_Gyro_Y', 'foot_Gyro_Z', 'shank_Gyro_X', 'shank_Gyro_Y', 'shank_Gyro_Z', 'thigh_Gyro_X', 'thigh_Gyro_Y', 'thigh_Gyro_Z', 'trunk_Gyro_X', 'trunk_Gyro_Y', 'trunk_Gyro_Z']
Treadmill force fields: ['Treadmill_L_vx', 'Treadmill_L_vy', 'Treadmill_L_vz', 'Treadmill_R_vx', 'Treadmill_R_vy', 'Treadmill_R_vz']


,subject_id,session_id,condition,trial_id,status,imu_start,imu_end,force_start,force_end,imu_median_step,force_median_step,native_overlap,start_difference,end_difference,read_error
0,AB06,10_09_18,treadmill,treadmill_01_01,readable_with_time_overlap,11.990,155.045,11.990,155.045,0.005,0.001,143.055,0.000,0.0,NaN
1,AB06,10_09_18,treadmill,treadmill_04_01,readable_with_time_overlap,12.475,156.035,12.472,156.035,0.005,0.001,143.560,0.003,0.0,NaN
2,AB06,10_09_18,treadmill,treadmill_08_01,readable_with_time_overlap,10.680,154.900,10.678,154.900,0.005,0.001,144.220,0.002,0.0,NaN
3,AB07,10_14_18,treadmill,treadmill_01_01,readable_with_time_overlap,10.360,153.415,10.359,153.415,0.005,0.001,143.055,0.001,0.0,NaN
4,AB07,10_14_18,treadmill,treadmill_04_2_01,readable_with_time_overlap,10.025,153.585,10.021,153.585,0.005,0.001,143.560,0.004,0.0,NaN
5,AB07,10_14_18,treadmill,treadmill_08_01,readable_with_time_overlap,8.795,153.015,8.793,153.015,0.005,0.001,144.220,0.002,0.0,NaN
6,AB08,10_21_2018,treadmill,treadmill_01_01,readable_with_time_overlap,8.630,151.525,8.627,151.525,0.005,0.001,142.895,0.003,0.0,NaN
7,AB08,10_21_2018,treadmill,treadmill_04_01,readable_with_time_overlap,12.085,155.470,12.082,155.470,0.005,0.001,143.385,0.003,0.0,NaN
8,AB08,10_21_2018,treadmill,treadmill_07_01,readable_with_time_overlap,9.575,153.455,9.573,153.455,0.005,0.001,143.880,0.002,0.0,NaN
9,AB09,10_21_2018,treadmill,treadmill_01_01,readable_with_time_overlap,14.020,156.920,14.017,156.920,0.005,0.001,142.900,0.003,0.0,NaN


,setting,candidate,observed,unresolved
0,GYRO_COLUMN,foot_Gyro_Y,True,Verify the selected gyroscope axis and instrum...
1,VERTICAL_FORCE_COLUMN,Treadmill_R_vy,True,"Verify physical vertical axis, source units an..."
2,FOOT_SIDE,right (candidate),<NA>,Confirm physical IMU foot side and force-plate...
3,FORCE_UPWARD_SIGN,positive (candidate),<NA>,Verify the real loading polarity and swing-pha...
4,TIME_UNIT_IN_SECONDS,1.0 (candidate),<NA>,Confirm Header time units with original 200/10...
5,IMU_FORCE_TIMEBASE_CONFIRMED,not confirmed,<NA>,Confirm a common event-level clock for retaine...


Descriptive source inspection complete. Contact-event extraction is still blocked pending physical channel and timebase verification.
Evidence saved to: /content/human-movement-intelligence/gait_analysis/output/02_source_contract_review.csv


**03. Extract Independent Force-Contact Reference Events**

Only after physical force-channel mapping, sign, foot-side correspondence, time unit and common event-level timing have been verified may this stage generate force-derived IC/TC labels. The Section 02 audit is descriptive evidence, **not** a reference-label validation.

If these settings remain unresolved, this stage stops deliberately with an explicit list. It does not reconstruct timestamps, invent contact labels or tune synchronisation against IMU predictions.


In [ ]:
# ====================== 03. Construct Force References =======================

# ------------------- 1. Validate Reviewed Source Contract --------------------
missing = [
    name for name, value in {
        "GYRO_COLUMN": GYRO_COLUMN,
        "VERTICAL_FORCE_COLUMN": VERTICAL_FORCE_COLUMN,
        "FOOT_SIDE": FOOT_SIDE if FOOT_SIDE in {"right", "left"} else None,
        "FORCE_UPWARD_SIGN": (
            FORCE_UPWARD_SIGN if FORCE_UPWARD_SIGN in (-1, 1) else None
        ),
        "TIME_UNIT_IN_SECONDS": TIME_UNIT_IN_SECONDS,
        "IMU_FORCE_TIMEBASE_CONFIRMED": IMU_FORCE_TIMEBASE_CONFIRMED,
    }.items() if value is None or value is False
]
if missing:
    raise ValueError(
        "Force-contact reference construction remains unavailable: "
        + ", ".join(missing)
        + ". Review 02_source_contract_review.csv and confirm the "
        "original physical channel, units, side and timing. Source-name "
        "candidates must not be reported as validated event references."
    )
if not np.isfinite(float(TIME_UNIT_IN_SECONDS)) or TIME_UNIT_IN_SECONDS <= 0:
    raise ValueError("Time unit conversion must be positive and finite.")
if GYRO_COLUMN not in imu_names or VERTICAL_FORCE_COLUMN not in fp_names:
    raise ValueError("Reviewed sensor channel absent from original data audit.")
missing_files = [
    source_root / relative
    for row in pairs.itertuples(index=False)
    for relative in (row.imu_path, row.fp_path)
    if not (source_root / relative).is_file()
]
if missing_files:
    raise FileNotFoundError(
        f"{len(missing_files)} original paired files are missing from "
        f"{source_root}. Use the same Colab runtime as Notebook 01, or "
        "rerun its acquisition step to restore source data."
    )

# ------------------------ 2. Extract Force References ------------------------
recorded, exclusions, event_rows = [], [], []
used_keys = set()
for row in pairs.itertuples(index=False):
    key = "_".join(re.sub(r"[^A-Za-z0-9]+", "_", str(v)).strip("_")
                   for v in (row.subject_id, row.session_id,
                             row.condition, row.trial_id))
    if key in used_keys:
        raise ValueError(f"Duplicate trial identity: {key}")
    used_keys.add(key)
    try:
        imu = read_sensor_table(source_root / row.imu_path)
        fp = read_sensor_table(source_root / row.fp_path)
        ti = validate_timebase(imu, "IMU") * float(TIME_UNIT_IN_SECONDS)
        tf = validate_timebase(fp, "force") * float(TIME_UNIT_IN_SECONDS)
        a, b = validated_overlap(ti, tf)
        gyro = mapped_series(imu, GYRO_COLUMN, "gyroscope")
        force = (mapped_series(fp, VERTICAL_FORCE_COLUMN, "vertical force")
                 * FORCE_UPWARD_SIGN)
        valid_imu = (ti >= a) & (ti <= b)
        valid_force = (tf >= a) & (tf <= b)
        events = force_contact_events(
            tf[valid_force], force[valid_force],
            threshold_n=REFERENCE_THRESHOLD_N
        )
        ic = events.loc[events.event_type == "IC", "t_reference_s"].to_numpy(float)
        if len(ic) < 3:
            raise ValueError("Too few complete independent force contacts.")
        save_prepared_trial(cache_dir, key, ti[valid_imu],
                            gyro[valid_imu], ic)
        recorded.append(dict(
            subject_id=row.subject_id, session_id=row.session_id,
            condition=row.condition, trial_id=row.trial_id, trial_key=key,
            side=FOOT_SIDE, gyro_channel=GYRO_COLUMN,
            vertical_force_channel=VERTICAL_FORCE_COLUMN,
            n_reference_ic=len(ic), duration_s=b-a
        ))
        for ev in events.itertuples(index=False):
            event_rows.append(dict(trial_key=key,
                subject_id=row.subject_id, event_type=ev.event_type,
                t_reference_s=ev.t_reference_s, side=FOOT_SIDE))
    except (ValueError, FileNotFoundError, KeyError) as error:
        exclusions.append(dict(subject_id=row.subject_id,
            session_id=row.session_id, condition=row.condition,
            trial_id=row.trial_id, exclusion_reason=str(error)))
eligible = pd.DataFrame(recorded, columns=[
    "subject_id","session_id","condition","trial_id","trial_key",
    "side","gyro_channel","vertical_force_channel","n_reference_ic","duration_s"])
excluded = pd.DataFrame(exclusions, columns=[
    "subject_id","session_id","condition","trial_id","exclusion_reason"])
references = pd.DataFrame(event_rows, columns=[
    "trial_key","subject_id","event_type","t_reference_s","side"])

# ------------------------- 2. Save Scientific Tables -------------------------
eligible.to_csv(output_dir / "02_eligible_trials.csv", index=False)
excluded.to_csv(output_dir / "02_exclusions.csv", index=False)
references.to_csv(output_dir / "02_reference_events.csv", index=False)
display(eligible.head(), excluded.head())
print("Accepted trials:", len(eligible),
      "Independent force IC events:", int((references.event_type == "IC").sum()))
if eligible.empty:
    raise ValueError(
        "No aligned eligible IMU/force trials. Review 02_exclusions.csv; "
        "do not calculate detector timing error without independent references."
    )


ValueError: Force-contact reference construction remains unavailable: GYRO_COLUMN, VERTICAL_FORCE_COLUMN, FOOT_SIDE, FORCE_UPWARD_SIGN, TIME_UNIT_IN_SECONDS, IMU_FORCE_TIMEBASE_CONFIRMED. Review 02_source_contract_review.csv and confirm the original physical channel, units, side and timing. Source-name candidates must not be reported as validated event references.

**04. Create Participant-Disjoint Development and Test Roles**

Reserve entire people before model tuning. The split size and available fold count depend on **observed eligible participants**, not an assumed fixed 22-person cohort. Pilot-scale runs with fewer than six people cannot be presented as final grouped evaluation.


In [ ]:
# ======================= 04. Assign Participant Roles ========================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
people = eligible.subject_id.unique()
if len(people) < 7:
    print("Source pilot complete for", len(people),
          "people. Download additional valid participants before grouped "
          "Development/Test assignment and Notebooks 04–05.")
    roles = pd.DataFrame(columns=["subject_id","role"])
    folds = pd.DataFrame(columns=["subject_id","validation_fold"])
else:
    roles = participant_roles(people, test_fraction=0.2, seed=42)
    folds = development_folds(roles, n_splits=5)
    print("Role counts:", roles.role.value_counts().to_dict())
    display(roles, folds)

# ------------------------- 2. Save Scientific Tables -------------------------
roles.to_csv(output_dir / "02_participant_roles.csv", index=False)
folds.to_csv(output_dir / "02_development_folds.csv", index=False)


**05. Inspect Source-Aligned Example and Document Limits**

Plot one eligible original trial with IMU and force-derived contact times, but do not convert an unverified time shift into an optimised offset. Reference IC events are an algorithmically derived measurement, not raw force observations.


In [ ]:
# ========================= 05. Inspect Aligned Trial =========================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
from gait_experiments import read_prepared_trial
sample = eligible.iloc[0]
t, g, ic = read_prepared_trial(cache_dir, sample.trial_key)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t, g, label=f"IMU: {GYRO_COLUMN}", lw=1)
for j, event in enumerate(ic):
    ax.axvline(event, color="darkorange", alpha=0.3,
               label="Force-derived IC" if j == 0 else None)
ax.set(xlabel="Reviewed original time (s)",
       ylabel="Gyroscope (source unit)", title="Eligible paired trial")
ax.legend()

# ------------------------ 2. Export Scientific Figure ------------------------
fig.tight_layout()
save_figure(fig, output_dir / "02_imu_reference_alignment.png")
plt.show()
print("Prepared numerical caches:", cache_dir)
print("Output reference events:", output_dir / "02_reference_events.csv")


**06. Research Summary and Output Download**

`02_source_contract_review.csv` is an original-source inspection output, not a completed force-reference analysis. Only running Sections 03–05 on eligible data can establish actual reference-event tables, exclusions, participant roles and aligned figures.

The final cell packages currently saved `02_` files into the study-specific ZIP. Raw Camargo recordings remain outside the ZIP and no GitHub push is performed.


In [ ]:
# ================= 06. Research Summary and Output Download ==================

# ------------------- 1. Review Current Scientific Outputs --------------------
print('Reference and eligibility evidence saved to:', output_dir)

# ------------------- 2. Package Notebook-Specific Evidence -------------------
download_outputs_from_colab(
    folder="gait_analysis/output",
    stage=f"{NOTEBOOK_NUMBER:02d}",
)
